# 2 · Structural characterization

Before generating any host–guest complexes, the isolated hosts and guests are
characterized with geometric and chemical descriptors. These provide the
quantitative basis for the correlation and selectivity analysis later (§7).

Implements methodology **§4**, documented in
[`modeling-and-simulation.md` §4](../modeling-and-simulation/modeling-and-simulation.md#4-structural-characterization).
No HPC resources are required.

| Subject | Tool | Output |
|---|---|---|
| Guest molecular descriptors | RDKit (via `mof-guest-toolkit`) | `thesis/040_DESCRIPTORS/guests_descriptors.csv` |
| Host pore geometry | [Zeo++ 0.3](https://www.zeoplusplus.org/) (via `mof-guest-toolkit`) | `thesis/040_DESCRIPTORS/hosts_descriptors.csv` |

## 4.1 Guest molecular descriptors (RDKit)

Five descriptors are computed for each of the 37 guests with the toolkit's
`mol_explorer` batch CLI, reading the same [`data/guests.csv`](../data/guests.csv)
used in system preparation:

| Descriptor | RDKit name |
|---|---|
| Molecular weight | `MolWt` |
| Rotatable bonds | `NumRotatableBonds` |
| H-bond donors | `NumHDonors` |
| H-bond acceptors | `NumHAcceptors` |
| Aromatic rings | `NumAromaticRings` |

In [ ]:
from pathlib import Path

from mof_toolkit.zeopp import batch_cif_to_cssr, run_zeo, collect_descriptors

# Base paths (relative to the repository root).
DATA   = Path("../data")      # reference inputs (guest list)
THESIS = Path("../thesis")    # numbered pipeline stages

DESCRIPTORS = THESIS / "040_DESCRIPTORS"
DESCRIPTORS.mkdir(parents=True, exist_ok=True)

In [ ]:
# RDKit descriptors for all 37 guests -> one CSV (toolkit CLI).
!mol_explorer -batch {DATA}/guests.csv \
    -descriptor MolWt NumRotatableBonds NumHDonors NumHAcceptors NumAromaticRings \
    -output {DESCRIPTORS}/guests_descriptors.csv

### 4.1 (cont.) Full descriptor set and 3D geometric descriptors

Two further descriptor tables are generated for all 37 guests and used in the
selectivity analysis. The **full RDKit 2D set** (`guests_descriptors2.csv`) is
produced with the same toolkit CLI using `-descriptor full`. The **3D geometric /
shape descriptors** (`guest_descriptors_3d.csv`) are computed directly from each
optimized guest geometry: the mass-weighted radius of gyration $R_g$, the
normalized principal-moment-of-inertia ratios NPR1 = $I_1/I_3$, NPR2 = $I_2/I_3$,
the long-axis extent, and the minimum/maximum van-der-Waals-inflated rotational
cross-section presented when the molecule is rotated about its principal long axis
(see [`modeling-and-simulation.md` §4](../modeling-and-simulation/modeling-and-simulation.md#4-structural-characterization) for the definitions).

In [ ]:
# Full RDKit 2D descriptor set (~200 descriptors) -> guests_descriptors2.csv
!mol_explorer -batch {DATA}/guests.csv -descriptor full \
    -output {DESCRIPTORS}/guests_descriptors2.csv

In [ ]:
# 3D geometric / shape descriptors from the optimized guest geometries.
import numpy as np, pandas as pd
from ase.io import read
from ase.data import vdw_radii, atomic_numbers

GUESTS_XYZ = THESIS / "012_GUESTS" / "final_structures"


def geometric_descriptors(xyz):
    atoms = read(xyz)
    r, m = atoms.get_positions(), atoms.get_masses()
    vdw = np.array([vdw_radii[atomic_numbers[s]] for s in atoms.get_chemical_symbols()])
    rc = r - (m[:, None] * r).sum(0) / m.sum()                         # centre of mass frame
    Rg = np.sqrt((m * (rc ** 2).sum(1)).sum() / m.sum())              # mass-weighted radius of gyration
    I = sum(mi * (np.eye(3) * (ri @ ri) - np.outer(ri, ri)) for mi, ri in zip(m, rc))
    ev, evec = np.linalg.eigh(I); o = np.argsort(ev); ev, evec = ev[o], evec[:, o]
    proj = rc @ evec                                                  # axis 0 = smallest moment = long axis
    long_extent = (proj[:, 0] + vdw).max() - (proj[:, 0] - vdw).min()
    widths = [(proj[:, 1:3] @ [np.cos(a), np.sin(a)] + vdw).max()
              - (proj[:, 1:3] @ [np.cos(a), np.sin(a)] - vdw).min()
              for a in np.linspace(0, np.pi, 180, endpoint=False)]    # rotating-calipers width
    return dict(n_atoms=len(r), Rg=Rg, NPR1=ev[0] / ev[2], NPR2=ev[1] / ev[2],
                long_axis_extent=long_extent, min_rot_width=min(widths), max_rot_width=max(widths))


family = dict(zip(pd.read_csv(DATA / "guests.csv").Abbreviation,
                  pd.read_csv(DATA / "guests.csv").Guest_Type))
rows = []
for f in sorted(GUESTS_XYZ.glob("*.xyz")):
    d = geometric_descriptors(f); d["Guest"] = f.stem; d["Family"] = family.get(f.stem); rows.append(d)
df3d = pd.DataFrame(rows).set_index("Guest")
df3d.to_csv(DESCRIPTORS / "guest_descriptors_3d.csv")
print("wrote guest_descriptors_3d.csv", df3d.shape)
df3d.round(2).head()

## 4.2 Host pore geometry (Zeo++)

Pore descriptors are computed for each GFN1-xTB-optimized host framework with
**Zeo++ 0.3**, which represents the void space of a periodic structure as a
Voronoi network. A probe radius of **1.55 Å** (≈ the kinetic radius of N₂) is
used throughout.

| Descriptor | Zeo++ flag |
|---|---|
| Pore-limiting & largest-cavity diameter | `-res` |
| Accessible surface area | `-sa` |
| Accessible volume | `-vol` |
| Probe-occupiable volume | `-volpo` |
| Pore size distribution | `-psd` |

The workflow has three steps — (1) convert each host CIF to CSSR, (2) run the
Zeo++ descriptor suite, (3) parse the outputs into one table — all wrapped in
`mof_toolkit.zeopp`.

### 1) Convert host CIFs to CSSR

Zeo++ reads CSSR (fractional coordinates, space group P1). `batch_cif_to_cssr`
converts every optimized host CIF in one call.

In [ ]:
hosts_cif_dir = THESIS / "011_HOSTS" / "final_structures"
cssr_dir = DESCRIPTORS / "hosts"

cssr_files = batch_cif_to_cssr(hosts_cif_dir, cssr_dir, pattern="*.cif")

### 2) Run Zeo++ and 3) collect the descriptors

`run_zeo` runs the full descriptor suite for each CSSR (writing a
`*_zeo_results/` folder per framework); `collect_descriptors` then parses all of
them into a single table saved as `hosts_descriptors.csv`.

> **Install Zeo++ separately** and point `ZEO_PATH` at its `network` binary.
> The `-psd` and `-volpo` calculations dominate the wall time (order of tens of
> minutes per framework).

In [ ]:
# Path to the Zeo++ `network` executable -- UPDATE for your machine.
ZEO_PATH = "/home/adriana/src/zeo++-0.3/network"

assert Path(ZEO_PATH).exists(), f"Zeo++ not found at {ZEO_PATH}; update ZEO_PATH."
print("Zeo++:", ZEO_PATH)

In [ ]:
# Run the full descriptor suite per framework (probe radius 1.55 Å by default).
run_zeo(cssr_files, ZEO_PATH)

In [ ]:
# Parse every *_zeo_results folder into one table and save it.
hosts_descriptors = collect_descriptors(
    cssr_dir,
    output_csv=DESCRIPTORS / "hosts_descriptors.csv",
    psd_output_dir=cssr_dir / "psd",
)
hosts_descriptors.round(3)

### 4.3 Host cluster coordination analysis

The carboxylate coordination mode of each Zr₆ cluster (Table 2 of the results) is
classified directly from the optimized host CIFs. Using a Zr–O cutoff of 2.9 Å and
an O–C cutoff of 1.65 Å under periodic boundary conditions, every carboxylate
carbon (a C bonded to exactly two O) is labelled **bridging** (its two oxygens
coordinate two *different* Zr) or **chelating** (both coordinate the *same* Zr).
The number of Zr₆ clusters per cell is the Zr count divided by six.

In [ ]:
# Carboxylate coordination per Zr6 cluster -> carboxylate_coordination.csv
import numpy as np, pandas as pd
from mof_toolkit.cif_tools import read_cif_symbols_frac_lattice

NETS = {"MOF-545": "csq", "NU-902": "scu", "PCN-223": "shp", "PCN-225": "sqc"}
ZR_O, O_C = 2.9, 1.65


def carboxylate_coordination(cif):
    sym, frac, L = read_cif_symbols_frac_lattice(cif)
    sym = np.array(sym); cart = (L @ frac.T).T; Linv = np.linalg.inv(L)
    def mind(d): df = Linv @ d; df -= np.round(df); return L @ df       # minimum image
    def nbr(i, cand, cut): return [j for j in cand if (lambda v: v @ v < cut ** 2)(mind(cart[j] - cart[i]))]
    zr = np.where(sym == "Zr")[0]; oo = np.where(sym == "O")[0]; cc = np.where(sym == "C")[0]
    bridging = chelating = 0
    for c in cc:
        os_ = nbr(c, oo, O_C)
        if len(os_) != 2:
            continue
        z0, z1 = set(nbr(os_[0], zr, ZR_O)), set(nbr(os_[1], zr, ZR_O))
        if not z0 or not z1:
            continue
        if z0 & z1:
            chelating += 1
        else:
            bridging += 1
    ncl = max(1, len(zr) // 6)
    return len(zr), ncl, bridging, chelating


rows = []
for f in sorted((THESIS / "011_HOSTS" / "final_structures").glob("*.cif")):
    mof = f.stem.replace("_dftb_geo-opt", "")
    nzr, ncl, br, ch = carboxylate_coordination(f)
    rows.append(dict(net=NETS.get(mof, mof), n_Zr_in_cell=nzr, n_clusters_in_cell=float(ncl),
                     carboxylates_per_cluster=(br + ch) / ncl, bridging_per_cluster=br / ncl,
                     chelating_per_cluster=ch / ncl, unclassified_per_cluster=0))
carbox = pd.DataFrame(rows)
carbox.to_csv(DESCRIPTORS / "carboxylate_coordination.csv", index=False)
print("wrote carboxylate_coordination.csv")
carbox

---

Both descriptor tables (`guests_descriptors.csv`, `hosts_descriptors.csv`) feed
the correlation analysis in
[`selectivity-analysis.ipynb`](./selectivity-analysis.ipynb) (§7).

**Next:** [`modeling-and-simulation.ipynb`](./modeling-and-simulation.ipynb)
— host–guest complex generation by stochastic rigid-body docking (SRD) and
molecular dynamics (MD).